# QGym training on Colab

Select the GPU type in the top right corner, then click Connect.

| Cell | Does |
|---|---|
| 1 | Mounts your Google Drive, clones the QGym repo, sets up necessary Python packages |
| 2 | Training |
| 3 | Saves trained NN to Drive |

# Make sure to disconnect the runtime before closing the tab or it will continue using your compute units!

In [9]:
# ============================== CELL 1: SETUP ==============================
BRANCH = "bh574/assignment_04"
REPO_URL = "https://github.com/TieneSabor/BigRedGym.git"
REPO = "/content/BigRedGym"
DRIVE = "/content/drive/MyDrive/BigRedGym"

import os, subprocess, textwrap

from google.colab import drive
drive.mount("/content/drive")
os.makedirs(f"{DRIVE}/checkpoints", exist_ok=True)
os.makedirs(f"{DRIVE}/warp_cache", exist_ok=True)

# uv installs to ~/.local/bin; warp caches compiled kernels on Drive.
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["WARP_CACHE_PATH"] = f"{DRIVE}/warp_cache"
os.environ["MPLBACKEND"] = "Agg"

setup = textwrap.dedent(f"""
    set -euo pipefail

    command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh

    if [ -d {REPO}/.git ]; then
        git -C {REPO} fetch origin {BRANCH}
        git -C {REPO} checkout {BRANCH}
        git -C {REPO} pull --ff-only
    else
        git clone --branch {BRANCH} {REPO_URL} {REPO}
    fi

    cd {REPO}
    uv python install 3.11          # pyproject pins requires-python == 3.11.*
    # --extra gpu pulls mujoco-warp (CUDA); --no-dev skips pytest/ruff/marimo.
    uv sync --frozen --extra gpu --no-dev || uv sync --extra gpu --no-dev
""")
subprocess.run(["bash", "-c", setup], check=True)

check = "import torch, mujoco, mujoco_warp; " \
        "print('torch', torch.__version__, '| mujoco', mujoco.__version__, " \
        "'| cuda', torch.cuda.is_available(), " \
        "'|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU')"
subprocess.run(
    ["uv", "run", "--frozen", "--extra", "gpu", "--no-dev", "python", "-c", check],
    cwd=REPO, check=True,
)
print("\nReady. If cuda is False, set Runtime -> Change runtime type -> GPU and re-run.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Ready. If cuda is False, set Runtime -> Change runtime type -> GPU and re-run.


In [ ]:
# ============================= CELL 2: TRAIN ==============================
TASK            = "go2trot"   # go2trot | mini_cheetah | humanoid | pendulum | ...
MAX_ITERATIONS  = 300
NUM_ENVS        = 4096
DEVICE          = "cuda:0"
EXPERIMENT_NAME = None        # None -> the task's default logs/<experiment_name>/
RESUME          = True        # True -> continue the newest run for this experiment
# RESUME          = False       # True -> continue the newest run for this experiment
USE_WANDB       = False       # True requires WANDB_API_KEY in the env (else it hangs)

REPO = "/content/BigRedGym"
DRIVE = "/content/drive/MyDrive/BigRedGym"

import os, subprocess, sys
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
os.environ["WARP_CACHE_PATH"] = f"{DRIVE}/warp_cache"
os.environ["MPLBACKEND"] = "Agg"

cmd = [
    "uv", "run", "--frozen", "--extra", "gpu", "--no-dev", "scripts/train.py",
    "--task", TASK,
    "--device", DEVICE,
    "--num_envs", str(NUM_ENVS),
    "--max_iterations", str(MAX_ITERATIONS),
    "--headless",
]
if EXPERIMENT_NAME:
    cmd += ["--experiment_name", EXPERIMENT_NAME]
if RESUME:
    cmd += ["--resume"]
if not USE_WANDB:
    cmd += ["--disable_wandb"]

print(" ".join(cmd), "\n", flush=True)
# First warp run JIT-compiles kernels (minutes, cached on Drive afterwards).
proc = subprocess.Popen(
    cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    text=True, bufsize=1,
)
for line in proc.stdout:
    print(line, end="")
if proc.wait() != 0:
    sys.exit(f"training exited with code {proc.returncode}")

uv run --frozen --extra gpu --no-dev scripts/train.py --task go2trot --device cuda:0 --num_envs 4096 --max_iterations 300 --headless --resume --disable_wandb 

Setting seed: 9655
Setting seed: 9655
Warp 1.16.0 initialized:
   CUDA Toolkit 12.9, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "Tesla T4" (15 GiB, sm_75, mempool enabled)
   Kernel cache:
     /content/drive/MyDrive/BigRedGym/warp_cache/1.16.0
Module mujoco_warp._src.io b4940be load on device 'cuda:0' took 11.13 ms  (cached)
Module mujoco_warp._src.smooth c4020f1 load on device 'cuda:0' took 14.75 ms  (cached)
Module _tile_cholesky_factorize_block__locals__kernel_dccc1fac 2d1dc66 load on device 'cuda:0' took 49.58 ms  (cached)
Module _tile_cholesky_solve_block__locals__kernel_cc0baf95 1fbe0d2 load on device 'cuda:0' took 52.75 ms  (cached)
Module _nxn_broadphase__locals__kernel_3e896ca5 3e896ca load on device 'cuda:0' took 3.77 ms  (cached)
Module ccd_kernel_builder__locals__ccd_kernel_c009c019 c009c01

In [13]:
# ==================== CELL 3: SAVE CHECKPOINTS TO DRIVE ====================
REPO = "/content/BigRedGym"
DRIVE = "/content/drive/MyDrive/BigRedGym"
ALL_RUNS = False   # False -> newest run only; True -> every run under logs/

import re, shutil
from pathlib import Path

def iteration(p):
    return int(re.search(r"model_(\d+)\.pt", p.name).group(1))

logs = Path(REPO) / "logs"
# Layout: logs/<experiment_name>/<Mon##_HH-MM-SS>_<run_name>/model_<iter>.pt
runs = sorted(
    {p.parent for p in logs.glob("*/*/model_*.pt")},
    key=lambda d: max(p.stat().st_mtime for p in d.glob("model_*.pt")),
)
if not runs:
    raise SystemExit(f"no checkpoints found under {logs}")
if not ALL_RUNS:
    runs = runs[-1:]

total_copied = total_bytes = total_skipped = 0
for run in runs:
    dest = Path(DRIVE) / "checkpoints" / run.parent.name / run.name
    dest.mkdir(parents=True, exist_ok=True)

    copied = []
    for ckpt in sorted(run.glob("model_*.pt"), key=iteration):
        target = dest / ckpt.name
        # Size match == already synced. Makes re-runs mid-training incremental,
        # so you can call this cell repeatedly without recopying gigabytes.
        if target.exists() and target.stat().st_size == ckpt.stat().st_size:
            total_skipped += 1
            continue
        shutil.copy2(ckpt, target)
        copied.append(ckpt)

    # The run's saved configs travel with it -- resume/play need them.
    for extra in run.iterdir():
        if extra.is_file() and not extra.name.startswith("model_"):
            shutil.copy2(extra, dest / extra.name)
        elif extra.is_dir():
            shutil.copytree(extra, dest / extra.name, dirs_exist_ok=True)

    nbytes = sum(c.stat().st_size for c in copied)
    total_copied += len(copied)
    total_bytes += nbytes
    print(f"{run.parent.name}/{run.name}: +{len(copied)} ckpt ({nbytes / 1e6:.1f} MB)")
    if copied:
        print(f"    iters {iteration(copied[0])}..{iteration(copied[-1])} -> {dest}")

print(f"\ncopied {total_copied} checkpoint(s), {total_bytes / 1e6:.1f} MB; "
      f"{total_skipped} already on Drive")

go2trot/Oct08_03-59-48_height-training: +7 ckpt (10.9 MB)
    iters 300..600 -> /content/drive/MyDrive/BigRedGym/checkpoints/go2trot/Oct08_03-59-48_height-training

copied 7 checkpoint(s), 10.9 MB; 0 already on Drive


# Disconnect the runtime before closing the tab or it will continue using your compute units!